# ChatModel+ChatPromptTemplate（输入与输出均为消息列表）
1、 模型接口：对应 LangChain 1.0 的主流接口 ChatModel。

2、 工作方式：现代聊天模型 API 已原生**支持角色概念**。它们不再接受单一字符串，而是要求输入**一个结构化的消息列表**。为构建复杂、可靠的多轮对话智能体系统奠定了坚实的基础。

3、Prompt 工具：**ChatPromptTemplate** 因此成为LangChain 1.0 中最核心的 Prompt 工具。它的职责是接收变量，并输出一个 List「BaseMessage」（消息列表），该列表可直接传递给聊天模型。

| 特性 | PromptTemplate | ChatPromptTemplate |
| ---- | -------------- | ------------------ |
| 输出格式 | 纯文本字符串 | 消息列表 |
| 角色支持 | ❌ 无 | ✅ system/user/assistant |
| 对话历史 | ❌ 不支持 | ✅ 支持 |
| 适用场景 | 简单提示 | 聊天、对话、多轮交互 |

ChatPromptTemplate，成为构建现代LangChain应用的首选工具。

核心理解：
1、支持消息列表输入
2、支持类型安全的变量替换
3、是调用ChatModel的前置操作，有实例化和调用两个步骤。调用通常就是给变量占位符赋值

## 实例初始化方法

1、使用类init方法直接初始化。和正常Python类的init方法相同。

2、使用内置类方法，如from_messages、from_template等。

**直接init**：不推荐，Python语言保留机制。

**from_messages**：推荐、常用，支持多种各种Python原始类型的列表初始化。如元组列表、字典列表、字符串列表等。

**from_template**：推荐，通常搭配MessagePromptTemplate使用，用于当使用消息类（BaseMessage的子类）列表实例化Template时，使得消息类可以支持变量替换。 


### 直接init

In [3]:
from langchain_core.prompts import ChatPromptTemplate
from rich import print as rich_print
#参数类型这里使用的是tuple构成的list
prompt_template = ChatPromptTemplate([
    # 字符串 role + 字符串 content
    ("system", "你是一个AI开发工程师. 你的名字是 {name}."),
    ("human", "你能开发哪些AI应用?"),
    ("ai", "我能开发很多AI应用, 比如聊天机器人, 图像识别, 自然语言处理等."),
    ("human", "{user_input}")
])

# 调用invoke()方法，返回ChatPromptValue，ChatPromptValue是调用模型的invoke()、stream()方法支持的输入格式

prompt = prompt_template.invoke({"name":"无敌AI", "user_input":"你能帮我做什么?"})
print(type(prompt))
rich_print(prompt)


<class 'langchain_core.prompt_values.ChatPromptValue'>


ChatPromptValue(
    messages=[
        SystemMessage(
            content='你是一个AI开发工程师. 你的名字是 无敌AI.',
            additional_kwargs={},
            response_metadata={}
        ),
        HumanMessage(content='你能开发哪些AI应用?', additional_kwargs={}, response_metadata={}),
        AIMessage(
            content='我能开发很多AI应用, 比如聊天机器人, 图像识别, 自然语言处理等.',
            additional_kwargs={},
            response_metadata={},
            tool_calls=[],
            invalid_tool_calls=[]
        ),
        HumanMessage(content='你能帮我做什么?', additional_kwargs={}, response_metadata={})
    ]
)

### 使用from_messages()方法

In [5]:

from langchain_core.prompts import ChatPromptTemplate
from rich import print as rich_print

chat_template = ChatPromptTemplate.from_messages(
    [
        ("system", "你是一个有帮助的AI机器人，你的名字是{name}。"),
        ("human", "你好，最近怎么样？"),
        ("ai", "我很好，谢谢！"),
        ("human", "{user_input}"),
    ]
)

prompt = chat_template.invoke({"name":"夸张AI", "user_input":"你能帮我做什么?"})
rich_print(prompt)


ChatPromptValue(
    messages=[
        SystemMessage(
            content='你是一个有帮助的AI机器人，你的名字是夸张AI。',
            additional_kwargs={},
            response_metadata={}
        ),
        HumanMessage(content='你好，最近怎么样？', additional_kwargs={}, response_metadata={}),
        AIMessage(
            content='我很好，谢谢！',
            additional_kwargs={},
            response_metadata={},
            tool_calls=[],
            invalid_tool_calls=[]
        ),
        HumanMessage(content='你能帮我做什么?', additional_kwargs={}, response_metadata={})
    ]
)

### 更多的初始参数

不管使用实例初始化方法，还是使用from_messages()，参数类型都是列表类型。列表中的元素可以是多种类型，前面我们主要测试了元组类型。

还支持：

```
MessageLike = BaseMessagePromptTemplate | BaseMessage | BaseChatPromptTemplate

MessageLikeRepresentation = (
    MessageLike
    | tuple[str | type, str | Sequence[dict[str, Any]] | Sequence[object]]
    | str
    | dict[str, Any]
)
```

参数是列表类型，列表的元素可以是:

字符串、字典、字符串构成的元组

消息类型、提示词模板类型、消息提示词模板类型等



#### str列表

列表参数格式是str类型（不推荐），因为默认角色都是human

In [22]:

from langchain_core.prompts import ChatPromptTemplate
from rich import print as rich_print
chat_template = ChatPromptTemplate.from_messages(
    [
        "Hello, {name}!",
        "what is the weather like today?"
    ]
)

messages = chat_template.invoke({"name":"小谷AI"})
rich_print(messages)

ChatPromptValue(
    messages=[
        HumanMessage(content='Hello, 小谷AI!', additional_kwargs={}, response_metadata={}),
        HumanMessage(content='what is the weather like today?', additional_kwargs={}, response_metadata={})
    ]
)

#### tuple列表类型

常用，此场景下语义固定，第一个元素是角色，第二个元素是消息内容。比字典更简洁。

In [25]:
prompt = ChatPromptTemplate.from_messages([
    ("system", "你是一个专业的翻译,你的名字是{name}"),
    ("human", "你好"),
    ("ai", "{response}")
])

prompt_value = prompt.format_messages(name="TEA——AI", response="你好")
rich_print(prompt_value)

[
    SystemMessage(content='你是一个专业的翻译,你的名字是TEA——AI', additional_kwargs={}, response_metadata={}),
    HumanMessage(content='你好', additional_kwargs={}, response_metadata={}),
    AIMessage(content='你好', additional_kwargs={}, response_metadata={}, tool_calls=[], invalid_tool_calls=[])
]

#### dict列表类型

和tuple列表类型类似，只是字典的键是角色，值是消息内容。需要额外指定角色。

In [26]:
prompt = ChatPromptTemplate.from_messages([
    {"role": "system", "content": "你是一个专业的翻译,你的名字是{name}"},
    {"role": "human", "content": "你好"},
    {"role": "ai", "content": "{response}"}
])

prompt_value = prompt.format_messages(name="天霸AI", response="你好")
rich_print(prompt_value)

[
    SystemMessage(content='你是一个专业的翻译,你的名字是天霸AI', additional_kwargs={}, response_metadata={}),
    HumanMessage(content='你好', additional_kwargs={}, response_metadata={}),
    AIMessage(content='你好', additional_kwargs={}, response_metadata={}, tool_calls=[], invalid_tool_calls=[])
]

#### Message列表与MessagePromptTemplate类型

使用LangChain的抽象消息类型，语义相对更清晰但是使用起来更复杂

- Message列表类型，不支持变量占位符
- MessagePromptTemplate类型，支持变量占位符，与Message类似有细分SystemMessagePromptTemplate、humanMessagePromptTemplate 和AIMessagePromptTemplate ，分别创建系统消息、人工消息和AI消息。

LangChain提供不同类型的MessagePromptTemplate。最常用的是`SystemMessagePromptTemplate`、`HumanMessagePromptTemplate`和`AIMessagePromptTemplate`，分别创建系统消息、人工消息和AI消息。

`HumanMessagePromptTemplate`，专用于生成**用户消息（HumanMessage）** 的模板类
- **模板化**：支持使用变量占位符，可以在运行时填充具体值
- **格式化**：能够将模板与输入变量结合生成最终的聊天消息
- **输出类型**：生成 `HumanMessage` 对象（`content` + `role="human"`）
- **设计目的**：简化用户输入消息的模板化构造，避免重复定义角色

`SystemMessagePromptTemplate`、`AIMessagePromptTemplate`：类似于上面，不再赘述

In [28]:
### Message列表类型
from langchain_core.messages import SystemMessage,HumanMessage
from rich import print as rich_print
chat_prompt_template = ChatPromptTemplate.from_messages([
    SystemMessage(content="我是一个贴心的智能助手"),
    HumanMessage(content="我的问题是:{word}英文怎么说？")
])
messages = chat_prompt_template.invoke({"word":"人工智能"})
rich_print(messages)
print(type(messages))

ChatPromptValue(
    messages=[
        SystemMessage(content='我是一个贴心的智能助手', additional_kwargs={}, response_metadata={}),
        HumanMessage(content='我的问题是:{word}英文怎么说？', additional_kwargs={}, response_metadata={})
    ]
)

<class 'langchain_core.prompt_values.ChatPromptValue'>


In [32]:
### MessagePromptTemplate类型
# 比较麻烦的点就在于各自的MessagePromptTemplate需要独立创建

# 导入聊天消息类模板
from langchain_core.prompts import ChatPromptTemplate, HumanMessagePromptTemplate, SystemMessagePromptTemplate
from rich import print as rich_print
# 创建各自类型消息模板
system_message_prompt = SystemMessagePromptTemplate.from_template("你是一个{role}")
human_message_prompt = HumanMessagePromptTemplate.from_template("给我解释{concept}，用浅显易懂的语言")

# 组合成聊天提示模板
chat_prompt = ChatPromptTemplate.from_messages([
    system_message_prompt,
    human_message_prompt
])

# 格式化提示
formatted_messages = chat_prompt.format_messages(role="物理学家", concept="相对论")
rich_print(formatted_messages)


[
    SystemMessage(content='你是一个物理学家', additional_kwargs={}, response_metadata={}),
    HumanMessage(content='给我解释相对论，用浅显易懂的语言', additional_kwargs={}, response_metadata={})
]

#### BaseChatPromptTemplate列表类型

使用 BaseChatPromptTemplate，可以理解为ChatPromptTemplate里嵌套了ChatPromptTemplate。


In [33]:
from langchain_core.prompts import ChatPromptTemplate
# 使用 BaseChatPromptTemplate（嵌套的 ChatPromptTemplate）
nested_prompt_template1 = ChatPromptTemplate.from_messages([
    ("system", "我是一个人工智能助手，我的名字叫{name}")
])
nested_prompt_template2 = ChatPromptTemplate.from_messages([
    ("human", "很高兴认识你,我的问题是{question}")
])
prompt_template = ChatPromptTemplate.from_messages([
    nested_prompt_template1,
    nested_prompt_template2
])
prompt_template.format_messages(name="小智", question="你为什么这么帅？")
rich_print(prompt_template.format_messages(name="小智", question="你为什么这么帅？"))


[
    SystemMessage(content='我是一个人工智能助手，我的名字叫小智', additional_kwargs={}, response_metadata={}),
    HumanMessage(content='很高兴认识你,我的问题是你为什么这么帅？', additional_kwargs={}, response_metadata={})
]

## 模板调用的3种方式

invoke()、format()、format_messages()

他们的主要区别是返回对象与变量赋值方法不同

invoke()：默认推荐，返回ChatPromptValue对象，变量赋值通过字典传递

format()：视情况使用，返回格式化后的字符串，变量赋值通过关键字参数传递

format_messages()：常用，返回格式化后的消息列表，变量赋值通过关键字参数传递

他们都可以作为模型调用的输入


### invoke()方法调用

In [14]:
from langchain_core.prompts import ChatPromptTemplate
from rich import print as rich_print
prompt_template = ChatPromptTemplate.from_messages([
    # 通常这里的语义明确，元组第一个元素是role，第二个元素是content，所以比字典更方便
    ("system", "你是一个AI开发工程师. 你的名字是 {name}."),
    ("human", "你能开发哪些AI应用?"),
    ("ai", "我能开发很多AI应用, 比如聊天机器人, 图像识别, 自然语言处理等."),
    ("human", "{user_input}")
])
prompt = prompt_template.invoke({"name":"KuiAI", "user_input":"你能帮我做什么?"})
print(type(prompt))
rich_print(prompt)
print(len(prompt.messages))

<class 'langchain_core.prompt_values.ChatPromptValue'>


ChatPromptValue(
    messages=[
        SystemMessage(
            content='你是一个AI开发工程师. 你的名字是 KuiAI.',
            additional_kwargs={},
            response_metadata={}
        ),
        HumanMessage(content='你能开发哪些AI应用?', additional_kwargs={}, response_metadata={}),
        AIMessage(
            content='我能开发很多AI应用, 比如聊天机器人, 图像识别, 自然语言处理等.',
            additional_kwargs={},
            response_metadata={},
            tool_calls=[],
            invalid_tool_calls=[]
        ),
        HumanMessage(content='你能帮我做什么?', additional_kwargs={}, response_metadata={})
    ]
)

4


### format()方法调用

In [15]:
from langchain_core.prompts import ChatPromptTemplate
from rich import print as rich_print
prompt_template = ChatPromptTemplate.from_messages([
    ("system", "你是一个AI开发工程师. 你的名字是 {name}."),
    ("human", "你能开发哪些AI应用?"),
    ("ai", "我能开发很多AI应用, 比如聊天机器人, 图像识别, 自然语言处理等."),
    ("human", "{user_input}")
])
prompt = prompt_template.format(name="KuiAI", user_input="你能帮我做什么?")
print(type(prompt))
rich_print(prompt)

<class 'str'>


System: 你是一个AI开发工程师. 你的名字是 KuiAI.
Human: 你能开发哪些AI应用?
AI: 我能开发很多AI应用, 比如聊天机器人, 图像识别, 自然语言处理等.
Human: 你能帮我做什么?

### format_messages()方法调用

In [16]:
from langchain_core.prompts import ChatPromptTemplate
from rich import print as rich_print
prompt_template = ChatPromptTemplate.from_messages([
    ("system", "你是一个AI开发工程师. 你的名字是 {name}."),
    ("human", "你能开发哪些AI应用?"),
    ("ai", "我能开发很多AI应用, 比如聊天机器人, 图像识别, 自然语言处理等."),
    ("human", "{user_input}")
])
prompt = prompt_template.format_messages(name="KuiAI", user_input="你能帮我做什么?")
print(type(prompt))
rich_print(prompt)
print(len(prompt))

<class 'list'>


[
    SystemMessage(content='你是一个AI开发工程师. 你的名字是 KuiAI.', additional_kwargs={}, response_metadata={}),
    HumanMessage(content='你能开发哪些AI应用?', additional_kwargs={}, response_metadata={}),
    AIMessage(
        content='我能开发很多AI应用, 比如聊天机器人, 图像识别, 自然语言处理等.',
        additional_kwargs={},
        response_metadata={},
        tool_calls=[],
        invalid_tool_calls=[]
    ),
    HumanMessage(content='你能帮我做什么?', additional_kwargs={}, response_metadata={})
]

4


## 搭配模型调用

作为模型调用的前置步骤

In [19]:
from dotenv import load_dotenv
from langchain_core.prompts import ChatPromptTemplate
import os
from langchain.chat_models import init_chat_model
from rich import print as rich_print

load_dotenv(override=True)

model = init_chat_model(
    model="gpt-5.4-mini",
    model_provider="openai",
    api_key=os.getenv("OPENROUTER_API_KEY"),
    base_url=os.getenv("OPENROUTER_BASE_URL")
)

# 定义提示模板
prompt_template = ChatPromptTemplate.from_messages([
    ("system", "你是一个小学数学老师，你的名字是 {name}，每次回答问题时必须以：”同学你好，{name}老师为你解答。“开头"),
    ("human", "{user_input}")
])

# 调用模板，可能从其他业务系统
prompt_value = prompt_template.format_messages(name="KICK", user_input="我不会计算10以上的加法，咋办。")

# 调用模型
response_blocks = model.invoke(prompt_value).content_blocks
rich_print(response_blocks)
answer = response_blocks[0].get("text")
print(answer)

[
    {
        'type': 'text',
        'text': '同学你好，KICK老师为你解答。\n\n不会算 10 以上的加法没关系，我们可以一步一步学，特别简单：\n\n### 
1. 先记住“凑十”\n比如：\n- 8 + 5  \n可以先想：8 再加 2 就到 10 了，5 里拿出 2，剩下 3  \n所以：8 + 5 = 10 + 3 = 
**13**\n\n### 2. 用手指数一数\n比如：\n- 7 + 6  \n先从 7 往后数 6 个：  \n8、9、10、11、12、13  \n所以答案是 
**13**\n\n### 3. 先从“10+几”开始\n比如：\n- 10 + 4 = 14\n- 10 + 7 = 17\n- 10 + 9 = 
19\n\n会了这个，再算别的就容易多了。\n\n### 4. 练习几个简单题\n你可以先试试：\n- 6 + 4 =\n- 8 + 2 =\n- 9 + 1 =\n- 7
+ 3 =\n\n这些都是“凑十”题，特别好练。\n\n如果你愿意，我可以马上带你练习 10 道“10以上加法”的题，我一题一题教你。'
    }
]

同学你好，KICK老师为你解答。

不会算 10 以上的加法没关系，我们可以一步一步学，特别简单：

### 1. 先记住“凑十”
比如：
- 8 + 5  
可以先想：8 再加 2 就到 10 了，5 里拿出 2，剩下 3  
所以：8 + 5 = 10 + 3 = **13**

### 2. 用手指数一数
比如：
- 7 + 6  
先从 7 往后数 6 个：  
8、9、10、11、12、13  
所以答案是 **13**

### 3. 先从“10+几”开始
比如：
- 10 + 4 = 14
- 10 + 7 = 17
- 10 + 9 = 19

会了这个，再算别的就容易多了。

### 4. 练习几个简单题
你可以先试试：
- 6 + 4 =
- 8 + 2 =
- 9 + 1 =
- 7 + 3 =

这些都是“凑十”题，特别好练。

如果你愿意，我可以马上带你练习 10 道“10以上加法”的题，我一题一题教你。


## 综合示例

In [35]:
from langchain_core.prompts import (
    ChatPromptTemplate,
    SystemMessagePromptTemplate,
    HumanMessagePromptTemplate,
)
from langchain_core.messages import SystemMessage, HumanMessage

# 示例 1: 使用 BaseMessage（已实例化的消息）
system_msg = SystemMessage(content="你是一个AI工程师。")
human_msg = HumanMessage(content="你好！")

# 示例 2: 使用 BaseMessagePromptTemplate
system_prompt = SystemMessagePromptTemplate.from_template("你是一个{role}.")
human_prompt = HumanMessagePromptTemplate.from_template("{user_input}")

# 示例 3: 使用 BaseChatPromptTemplate（嵌套的 ChatPromptTemplate）
nested_prompt = ChatPromptTemplate.from_messages([("system", "嵌套提示词")])

prompt = ChatPromptTemplate.from_messages([
    system_msg,     # MessageLike (BaseMessage)
    human_msg,      # MessageLike (BaseMessage)
    system_prompt,  # MessageLike (BaseMessagePromptTemplate)
    human_prompt,   # MessageLike (BaseMessagePromptTemplate)
    nested_prompt,  # MessageLike (BaseChatPromptTemplate)
])

prompt.format_messages(role="人工智能专家", user_input="介绍一下大模型的应用场景")
rich_print(prompt.format_messages(role="人工智能专家", user_input="介绍一下大模型的应用场景"))


[
    SystemMessage(content='你是一个AI工程师。', additional_kwargs={}, response_metadata={}),
    HumanMessage(content='你好！', additional_kwargs={}, response_metadata={}),
    SystemMessage(content='你是一个人工智能专家.', additional_kwargs={}, response_metadata={}),
    HumanMessage(content='介绍一下大模型的应用场景', additional_kwargs={}, response_metadata={}),
    SystemMessage(content='嵌套提示词', additional_kwargs={}, response_metadata={})
]

## 查漏补缺：项目里常用但容易漏的点

前面的示例已经覆盖了 `ChatPromptTemplate` 的主要初始化方式和三种格式化方法。下面补几个实际项目里更常见、也更容易混淆的用法。


### 1. 单变量模板可以直接传值

当模板里只有一个输入变量时，`invoke()` 可以直接传入普通值。LangChain 会自动把这个值填到唯一的变量位置。

这种写法适合非常简单的模板；如果变量超过一个，仍然必须传字典。


In [ ]:
from langchain_core.prompts import ChatPromptTemplate
from rich import print as rich_print

prompt = ChatPromptTemplate.from_messages([
    ("system", "你是一个回答简洁的助手。"),
    ("human", "{question}"),
])

# 因为模板中只有 question 一个变量，所以可以直接传字符串。
prompt_value = prompt.invoke("ChatPromptTemplate 是什么？")

rich_print(prompt_value)
print(prompt.input_variables)


### 2. MessagesPlaceholder：插入一段已有对话历史

`MessagesPlaceholder` 用来把“已经存在的一组消息”插入到模板的某个位置。它不是把历史消息拼成一个字符串，而是保留每条消息自己的角色。

这在多轮对话、带 checkpoint 的 agent、聊天机器人里很常见。


In [ ]:
from langchain_core.prompts import ChatPromptTemplate, MessagesPlaceholder
from rich import print as rich_print

prompt = ChatPromptTemplate.from_messages([
    ("system", "你是一个耐心的 Python 老师。"),
    MessagesPlaceholder("history"),
    ("human", "{question}"),
])

prompt_value = prompt.invoke({
    "history": [
        ("human", "我刚开始学 Python。"),
        ("ai", "没问题，我会尽量用初学者能理解的方式解释。"),
    ],
    "question": "什么是列表推导式？",
})

rich_print(prompt_value.messages)


### 3. 可选历史消息与最近 N 条消息

如果历史消息不是每次都有，可以设置 `optional=True`。这样没有传 `history` 时不会报错，而是插入空列表。

`n_messages` 可以限制最多插入最近几条消息，避免历史对话过长。


In [ ]:
from langchain_core.prompts import ChatPromptTemplate, MessagesPlaceholder
from rich import print as rich_print

prompt = ChatPromptTemplate.from_messages([
    ("system", "你是一个简洁的助手。"),
    MessagesPlaceholder("history", optional=True, n_messages=2),
    ("human", "{question}"),
])

# 没有传 history 也可以正常运行。
rich_print(prompt.invoke({"question": "你好"}).messages)

# 传入多条历史消息时，只保留最近 2 条。
rich_print(prompt.invoke({
    "history": [
        ("human", "第一轮问题"),
        ("ai", "第一轮回答"),
        ("human", "第二轮问题"),
        ("ai", "第二轮回答"),
    ],
    "question": "继续解释一下",
}).messages)


### 4. placeholder

如果不想显式导入 `MessagesPlaceholder`，也可以用 `("placeholder", "{history}")` 简写。

这个简写默认是可选占位符，适合快速写聊天历史模板。需要 `n_messages` 等高级参数时，使用 `MessagesPlaceholder(...)` 更清晰。


In [ ]:
# 简写模式

from langchain_core.prompts import ChatPromptTemplate
from rich import print as rich_print

prompt = ChatPromptTemplate.from_messages([
    ("system", "你是一个代码助手。"),
    ("placeholder", "{history}"),
    ("human", "{question}"),
])

rich_print(prompt.invoke({
    "history": [("human", "我在学 LangChain。")],
    "question": "ChatPromptTemplate 主要解决什么问题？",
}).messages)


In [ ]:
# MessagesPlaceholder，简写方法就是JSON格式，Key是placeholder，Value是变量名

from langchain_core.prompts import ChatPromptTemplate, MessagesPlaceholder
prompt_template = ChatPromptTemplate.from_messages(
    [
        ("system", "你是一个非常友好的AI助手"),
        MessagesPlaceholder(variable_name="history"),
        ("human", "{question}")
    ]
)
prompt_template.invoke(
    {
        "history": [
            ("human", "5 + 2 = ?"), # 也可以是HumanMessage
            ("ai", "5 + 2 = 7")
        ],
        "question": "结果再乘以4呢？"
    }
)

ChatPromptValue(messages=[SystemMessage(content='你是一个非常友好的AI助手', additional_kwargs={}, response_metadata={}), HumanMessage(content='5 + 2 = ?', additional_kwargs={}, response_metadata={}), AIMessage(content='5 + 2 = 7', additional_kwargs={}, response_metadata={}, tool_calls=[], invalid_tool_calls=[]), HumanMessage(content='结果再乘以4呢？', additional_kwargs={}, response_metadata={})])

### 5. partial：提前固定一部分变量

`partial()` 可以把某些变量先填好，得到一个新的模板。后续调用时只需要传剩下的变量。

常见用途：固定角色、语言、系统规则、日期、业务名称等上下文。


In [ ]:
from langchain_core.prompts import ChatPromptTemplate
from rich import print as rich_print

prompt = ChatPromptTemplate.from_messages([
    ("system", "你是一个{role}，请使用{language}回答。"),
    ("human", "{question}"),
])

# 先固定 role 和 language，后续只需要传 question。
python_teacher_prompt = prompt.partial(role="Python 老师", language="中文")

rich_print(python_teacher_prompt.invoke({
    "question": "解释一下装饰器是什么。"
}))
print(python_teacher_prompt.input_variables)


In [39]:
from langchain_core.prompts import ChatPromptTemplate
from rich import print as rich_print

base_template = ChatPromptTemplate.from_messages([
    ("system", "你是{department}的{role}"),
    ("user", "{task}")
])

# IT 部门
it_template = base_template.partial(
    department="IT 部门",
    role="技术支持"
)

# 销售部门
sales_template = base_template.partial(
    department="销售部门",
    role="销售顾问"
)
it_result = it_template.invoke({"task": "请介绍一下大模型的应用场景"})
rich_print(it_result)
sales_result = sales_template.invoke({"task": "为什么每年年底汽车会促销"})
rich_print(sales_result)


ChatPromptValue(
    messages=[
        SystemMessage(content='你是IT 部门的技术支持', additional_kwargs={}, response_metadata={}),
        HumanMessage(content='请介绍一下大模型的应用场景', additional_kwargs={}, response_metadata={})
    ]
)

ChatPromptValue(
    messages=[
        SystemMessage(content='你是销售部门的销售顾问', additional_kwargs={}, response_metadata={}),
        HumanMessage(content='为什么每年年底汽车会促销', additional_kwargs={}, response_metadata={})
    ]
)

### 6. 关于 LCEL 管道写法：了解即可

`prompt | model` 是 LCEL / Runnable 管道写法，不是旧版 `LLMChain` / `ConversationChain` 那类 legacy Chain。

它目前仍然可用，官方文档在 LangSmith tracing、评测、prompt hub 等场景里还会使用这种写法。但在 LangChain v1 的主线应用开发里，复杂应用通常更推荐 `create_agent` / LangGraph / middleware 这类结构。

所以本节只需要知道：`ChatPromptTemplate` 本身也是 Runnable，可以参与管道组合；学习提示词模板时，优先掌握 `prompt.invoke()` 和 `prompt.format_messages()` 即可。


In [36]:
from langchain_core.prompts import ChatPromptTemplate
from rich import print as rich_print

prompt = ChatPromptTemplate.from_messages([
    ("system", "你是一个小学数学老师，回答要简短清楚。"),
    ("human", "{question}"),
])

# prompt 本身是 Runnable，所以可以和其他 Runnable 组合。
# 这里不展开模型调用，只观察 prompt.invoke() 的结果。
prompt_value = prompt.invoke({"question": "为什么 8 + 7 = 15？"})
rich_print(prompt_value.messages)


[
    SystemMessage(content='你是一个小学数学老师，回答要简短清楚。', additional_kwargs={}, response_metadata={}),
    HumanMessage(content='为什么 8 + 7 = 15？', additional_kwargs={}, response_metadata={})
]

### 7. 模板里如果要保留字面量大括号，需要转义

`ChatPromptTemplate` 默认使用 f-string 风格模板，`{name}` 会被识别为变量。

如果你想在提示词里展示 JSON、字典、集合等包含 `{}` 的内容，需要用双大括号 `{{` 和 `}}` 表示字面量大括号。


In [ ]:
from langchain_core.prompts import ChatPromptTemplate
from rich import print as rich_print

prompt = ChatPromptTemplate.from_messages([
    ("system", "你是一个 JSON 助手。"),
    ("human", '请按这个 JSON 格式回答：{{"answer": "你的答案"}}。问题：{question}'),
])

rich_print(prompt.invoke({"question": "Python 是什么？"}).messages)


## 模板库用法

有各种不同类型的模板，可以用文件单独存储

在template.py中

```python
from langchain_core.prompts import ChatPromptTemplate
class PromptLibrary:
    """可复用的提示词模板库"""
    TRANSLATOR = ChatPromptTemplate.from_messages([
        ("system", "你是专业翻译，精通{source_lang}和{target_lang}"),
        ("user", "翻译以下文本：\n{text}")
    ])
    CODE_REVIEWER = ChatPromptTemplate.from_messages([
        ("system", "你是{language}代码审查专家，重点关注{focus}"),
        ("user", "审查代码：\n```{language}\n{code}\n```")
    ])
    SUMMARIZER = ChatPromptTemplate.from_messages([
        ("system", "你是内容摘要专家"),
        ("user", "将以下内容总结为{num}个要点：\n{content}")
    ])
    TUTOR = ChatPromptTemplate.from_messages([
        ("system", "你是{subject}导师，学生水平：{level}"),
        ("user", "{question}")
    ])
```


其他地方使用

```python
from templates import PromptLibrary
messages = PromptLibrary.TRANSLATOR.format_messages(
    source_lang="英语",
    target_lang="中文",
    text="Hello World"
)


template1 = ChatPromptTemplate.from_messages([
    ("system", "你是助手")
])
template2 = ChatPromptTemplate.from_messages([
    ("user", "{input}")
])
# 组合（LangChain 1.0 支持）
combined = template1 + template2
```

